# Gemma-Mitra Embedding Protocol Comparison

This notebook compares four sentence-embedding protocols on the same two Tibetan passages:

- asymmetric `A -> B` retrieval mode (`query_corpus`)
- asymmetric `B -> A` retrieval mode (`query_corpus`, reversed)
- symmetric `query_query`
- symmetric `raw_raw`

The goal is to make the protocol choice inspectable before scaling to corpus-level analysis.

In [1]:
!pip install --no-build-isolation pyewts botok
!pip install -r /content/embedding-model-for-tibetan/requirements.txt


In [2]:
!ls -la

total 24
drwxr-xr-x 1 root root 4096 Apr 26 22:50 .
drwxr-xr-x 1 root root 4096 Apr 26 22:40 ..
drwxr-xr-x 3 root root 4096 Apr 26 22:50 .cache
drwxr-xr-x 1 root root 4096 Mar 30 13:34 .config
drwxr-xr-x 9 root root 4096 Apr 26 22:44 embedding-model-for-tibetan
drwxr-xr-x 1 root root 4096 Mar 30 13:34 sample_data


In [3]:
!nvidia-smi
import torch

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu_name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("total_vram_gb:", round(props.total_memory / 1024**3, 2))


Sun Apr 26 23:02:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       2.9Gi       2.6Gi       2.0Mi       7.2Gi       9.5Gi
Swap:             0B          0B          0B


In [5]:
!pip install -U bitsandbytes accelerate


In [6]:
import os
import subprocess
import sys
from pathlib import Path

AUTO_INSTALL_DEPS = True
REPO_URL = "https://github.com/ten-jampa/embedding-model-for-tibetan.git"
COLAB_REPO_DIR = Path("/content/embedding-model-for-tibetan")


def running_in_colab() -> bool:
    return "google.colab" in sys.modules


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "tibetan_pipeline").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find project root containing tibetan_pipeline/ and README.md")


def ensure_repo() -> Path:
    env_root = os.environ.get("TIBETAN_PIPELINE_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if candidate.exists():
            return find_project_root(candidate)

    if running_in_colab():
        if not COLAB_REPO_DIR.exists():
            subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
        return find_project_root(COLAB_REPO_DIR)

    return find_project_root(Path.cwd())


def install_runtime_deps(project_root: Path) -> None:
    requirements_path = project_root / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "pyewts", "botok"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], check=True)


PROJECT_ROOT = ensure_repo()
if AUTO_INSTALL_DEPS:
    install_runtime_deps(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"PYTHON={sys.executable}")
print(f"IN_COLAB={running_in_colab()}")
print("Note: Colab clone uses origin/main unless you point TIBETAN_PIPELINE_PROJECT_ROOT at another checkout.")

from tibetan_pipeline import TibetanResearchSDK
from tibetan_pipeline.sdk import EmbeddingView


PROJECT_ROOT=/content/embedding-model-for-tibetan
PYTHON=/usr/bin/python3
IN_COLAB=True
Note: Colab clone uses origin/main unless you point TIBETAN_PIPELINE_PROJECT_ROOT at another checkout.


## Using 8bit model

In [7]:
import numpy as np
import pandas as pd

from tibetan_pipeline import TibetanResearchSDK

sdk = TibetanResearchSDK(
    engine="botok_ours",
    source_format="unicode",
    model_id="buddhist-nlp/gemma-2-mitra-e",
    device="cuda",
    batch_size=1,
    embedding_progress="batch",
    torch_dtype="float16",
    load_in_8bit=True,
)


In [8]:
print("device:", sdk.device)
print("torch_dtype:", sdk.torch_dtype)
print("load_in_8bit:", sdk.load_in_8bit)
print("batch_size:", sdk.batch_size)


device: cuda
torch_dtype: float16
load_in_8bit: True
batch_size: 1


In [9]:
from time import perf_counter

t0 = perf_counter()
test_emb = sdk.embed_sentences(["ང་བོད་པ་ཡིན་"], is_query=True)
t1 = perf_counter()
print("warmup_seconds:", round(t1 - t0, 2))
print("shape:", test_emb.embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

[embed] backend=gemma-last-token device=cuda total_sentences=1 batch_size=1 batches=1
[embed] batch 1/1 sentences 1-1/1
warmup_seconds: 152.88
shape: (1, 3584)


In [13]:
import time

t0 = time.perf_counter()
test_emb2 = sdk.embed_sentences(["ང་བོད་ལ་དགའ་"], is_query=False)
t1 = time.perf_counter()

print("second_call_seconds:", round(t1 - t0, 2))
print("shape:", test_emb2.embeddings.shape)


[embed] backend=gemma-last-token device=cuda total_sentences=1 batch_size=1 batches=1
[embed] batch 1/1 sentences 1-1/1
second_call_seconds: 0.79
shape: (1, 3584)


In [21]:
import torch
print(torch.cuda.memory_allocated() / 1024**3)
print(torch.cuda.memory_reserved() / 1024**3)


9.479180335998535
9.80859375


## Input Passages

Replace these with any two Tibetan paragraphs you want to study. Keep them as multi-sentence passages so the notebook can expose the sentence-level matrix behavior.

In [14]:
text_a = """
རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚོས་མཆོད་པ་བསྒྲིགས། ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས་སྒོམ་པ་བྱས། མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐོས་པ་བྱུང་།
""".strip()

text_b = """
བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་མཆོད་པ་བཤམས། ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛིན་བསྐྱངས། མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམས་ཞི་བར་གྱུར།
""".strip()

print(text_a)
print()
print(text_b)

རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚོས་མཆོད་པ་བསྒྲིགས། ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས་སྒོམ་པ་བྱས། མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐོས་པ་བྱུང་།

བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་མཆོད་པ་བཤམས། ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛིན་བསྐྱངས། མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམས་ཞི་བར་གྱུར།


In [15]:
seg_a = sdk.segment_text(text_a)
seg_b = sdk.segment_text(text_b)

segments_a = [segment for segment in seg_a.segments if segment.strip()]
segments_b = [segment for segment in seg_b.segments if segment.strip()]

segment_overview = pd.DataFrame(
    [
        {
            "text": "A",
            "segment_count": len(segments_a),
            "max_segment_chars": max((len(s) for s in segments_a), default=0),
            "mean_segment_chars": round(np.mean([len(s) for s in segments_a]), 2) if segments_a else 0,
        },
        {
            "text": "B",
            "segment_count": len(segments_b),
            "max_segment_chars": max((len(s) for s in segments_b), default=0),
            "mean_segment_chars": round(np.mean([len(s) for s in segments_b]), 2) if segments_b else 0,
        },
    ]
)

display(segment_overview)
display(pd.DataFrame({"A_segments": pd.Series(segments_a), "B_segments": pd.Series(segments_b)}))

,text,segment_count,max_segment_chars,mean_segment_chars
0,A,3,64,59.67
1,B,3,59,57.33


,A_segments,B_segments
0,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
1,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
2,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...


In [16]:
def pairwise_protocol(protocol_name: str, sentences_a: list[str], sentences_b: list[str]):
    if protocol_name == "query_corpus":
        emb_a = sdk.embed_sentences(sentences_a, is_query=True)
        emb_b = sdk.embed_sentences(sentences_b, is_query=False)
    elif protocol_name == "query_query":
        emb_a = sdk.embed_sentences(sentences_a, is_query=True)
        emb_b = sdk.embed_sentences(sentences_b, is_query=True)
    elif protocol_name == "raw_raw":
        emb_a = sdk.embed_sentences(sentences_a, is_query=False)
        emb_b = sdk.embed_sentences(sentences_b, is_query=False)
    else:
        raise ValueError(f"Unsupported protocol: {protocol_name}")

    return sdk.pairwise_from_embedding_views(emb_a, emb_b, top_k=min(10, len(sentences_a) * len(sentences_b)))


def summarize_view(label: str, view) -> dict[str, object]:
    matrix = view.similarity_matrix
    return {
        "label": label,
        "shape": tuple(matrix.shape),
        "max_score": round(float(np.max(matrix)), 4) if matrix.size else None,
        "mean_score": round(float(np.mean(matrix)), 4) if matrix.size else None,
        "p95_score": round(float(np.percentile(matrix, 95)), 4) if matrix.size else None,
        "mean_best_row": round(float(np.mean(np.max(matrix, axis=1))), 4) if matrix.size else None,
        "mean_best_col": round(float(np.mean(np.max(matrix, axis=0))), 4) if matrix.size else None,
    }


def matrix_frame(view, row_label: str, col_label: str) -> pd.DataFrame:
    return pd.DataFrame(
        view.similarity_matrix,
        index=[f"{row_label}{i:02d}" for i in range(len(view.segments_a))],
        columns=[f"{col_label}{j:02d}" for j in range(len(view.segments_b))],
    )


In [17]:
view_ab = pairwise_protocol("query_corpus", segments_a, segments_b)
view_ba = pairwise_protocol("query_corpus", segments_b, segments_a)
view_qq = pairwise_protocol("query_query", segments_a, segments_b)
view_rr = pairwise_protocol("raw_raw", segments_a, segments_b)

summary_df = pd.DataFrame(
    [
        summarize_view("A->B query_corpus", view_ab),
        summarize_view("B->A query_corpus", view_ba),
        summarize_view("A<->B query_query", view_qq),
        summarize_view("A<->B raw_raw", view_rr),
    ]
)

display(summary_df)

[embed] backend=gemma-last-token device=cuda total_sentences=3 batch_size=1 batches=3
[embed] batch 1/3 sentences 1-1/3
[embed] batch 2/3 sentences 2-2/3
[embed] batch 3/3 sentences 3-3/3
[embed] backend=gemma-last-token device=cuda total_sentences=3 batch_size=1 batches=3
[embed] batch 1/3 sentences 1-1/3
[embed] batch 2/3 sentences 2-2/3
[embed] batch 3/3 sentences 3-3/3
[embed] backend=gemma-last-token device=cuda total_sentences=3 batch_size=1 batches=3
[embed] batch 1/3 sentences 1-1/3
[embed] batch 2/3 sentences 2-2/3
[embed] batch 3/3 sentences 3-3/3
[embed] backend=gemma-last-token device=cuda total_sentences=3 batch_size=1 batches=3
[embed] batch 1/3 sentences 1-1/3
[embed] batch 2/3 sentences 2-2/3
[embed] batch 3/3 sentences 3-3/3
[embed] backend=gemma-last-token device=cuda total_sentences=3 batch_size=1 batches=3
[embed] batch 1/3 sentences 1-1/3
[embed] batch 2/3 sentences 2-2/3
[embed] batch 3/3 sentences 3-3/3
[embed] backend=gemma-last-token device=cuda total_sentences

,label,shape,max_score,mean_score,p95_score,mean_best_row,mean_best_col
0,A->B query_corpus,"(3, 3)",0.7586,0.4951,0.7467,0.7234,0.7234
1,B->A query_corpus,"(3, 3)",0.7636,0.5047,0.7577,0.7360,0.7360
2,A<->B query_query,"(3, 3)",0.8143,0.5430,0.8000,0.7771,0.7771
3,A<->B raw_raw,"(3, 3)",0.8078,0.5382,0.8060,0.7800,0.7800


## Similarity Matrices

Look for two things:

- whether `A -> B` and `B -> A` are materially different
- whether the symmetric protocols preserve the same strongest alignments

In [18]:
display(matrix_frame(view_ab, "A", "B").round(4))
display(matrix_frame(view_ba, "B", "A").round(4))
display(matrix_frame(view_qq, "A", "B").round(4))
display(matrix_frame(view_rr, "A", "B").round(4))

,B00,B01,B02
A00,0.7586,0.4143,0.3951
A01,0.4115,0.6828,0.3473
A02,0.3709,0.3467,0.7288


,A00,A01,A02
B00,0.7636,0.4471,0.4038
B01,0.4028,0.6957,0.3367
B02,0.3701,0.3734,0.7487


,B00,B01,B02
A00,0.8143,0.4601,0.4269
A01,0.4711,0.7384,0.3964
A02,0.4291,0.3724,0.7785


,B00,B01,B02
A00,0.8033,0.4325,0.4116
A01,0.4517,0.7288,0.3920
A02,0.4250,0.3907,0.8078


In [19]:
topk_tables = {
    "A->B query_corpus": view_ab.topk_dataframe(),
    "B->A query_corpus": view_ba.topk_dataframe(),
    "A<->B query_query": view_qq.topk_dataframe(),
    "A<->B raw_raw": view_rr.topk_dataframe(),
}

for label, table in topk_tables.items():
    print(f"## {label}")
    display(table.head(10))

## A->B query_corpus


,rank,score,i,j,sentence_a,sentence_b
0,1,0.758572,0,0,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
1,2,0.728814,2,2,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
2,3,0.682754,1,1,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
3,4,0.414284,0,1,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
4,5,0.411482,1,0,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
5,6,0.395080,0,2,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
6,7,0.370906,2,0,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
7,8,0.347286,1,2,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
8,9,0.346737,2,1,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...


## B->A query_corpus


,rank,score,i,j,sentence_a,sentence_b
0,1,0.763627,0,0,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...
1,2,0.748733,2,2,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...
2,3,0.695712,1,1,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...
3,4,0.447115,0,1,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...
4,5,0.403808,0,2,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...
5,6,0.402819,1,0,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...
6,7,0.373356,2,1,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...
7,8,0.370057,2,0,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...
8,9,0.336742,1,2,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...


## A<->B query_query


,rank,score,i,j,sentence_a,sentence_b
0,1,0.814318,0,0,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
1,2,0.778494,2,2,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
2,3,0.738439,1,1,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
3,4,0.471087,1,0,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
4,5,0.460115,0,1,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
5,6,0.429080,2,0,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
6,7,0.426863,0,2,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
7,8,0.396416,1,2,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
8,9,0.372376,2,1,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...


## A<->B raw_raw


,rank,score,i,j,sentence_a,sentence_b
0,1,0.807833,2,2,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
1,2,0.803327,0,0,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
2,3,0.728774,1,1,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
3,4,0.451708,1,0,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
4,5,0.432526,0,1,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...
5,6,0.424955,2,0,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,བྲག་ཕུག་གི་ཉེ་འཁོར་དུ་སྒོམ་ཆེན་ཞིག་གིས་སྔ་དྲོ་...
6,7,0.411612,0,2,རི་ཁྲོད་ཀྱི་དགོན་པ་དེར་སྔ་དྲོ་ཞིག་ལ་དགེ་སློང་ཚ...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
7,8,0.392034,1,2,ཁོང་ཚོས་བཀའ་འགྱུར་གྱི་ཚིག་དོན་ལ་ཞིབ་ཏུ་བསམས་ནས...,མཚན་མོ་ནས་ཆུ་དང་རླུང་གི་སྒྲ་མཉམ་དུ་གྲགས་པས་སེམ...
8,9,0.390660,2,1,མཚན་མོར་རླུང་གི་སྒྲ་དང་ཆུ་ཡི་སྒྲ་གཉིས་མཉམ་དུ་ཐ...,ཁོས་གསུང་རབ་ཀྱི་དོན་ལ་ཡང་ཡང་བསམས་ནས་ཏིང་ངེ་འཛི...


In [20]:
comparison = summary_df.set_index("label")[["max_score", "mean_score", "p95_score", "mean_best_row", "mean_best_col"]]
comparison

,max_score,mean_score,p95_score,mean_best_row,mean_best_col
label,,,,,
A->B query_corpus,0.7586,0.4951,0.7467,0.7234,0.7234
B->A query_corpus,0.7636,0.5047,0.7577,0.7360,0.7360
A<->B query_query,0.8143,0.5430,0.8000,0.7771,0.7771
A<->B raw_raw,0.8078,0.5382,0.8060,0.7800,0.7800


## Notes for Interpretation on this limited set

- `A -> B query_corpus` follows the current model-card retrieval usage.
- `B -> A query_corpus` tests whether reversing the query/corpus roles changes the result materially.
- `query_query` removes the role mismatch by wrapping both sides as queries.
- `raw_raw` removes the instruction wrapper and treats both sides as plain text.

If the top-ranked sentence pairs are stable across all four views, the later corpus-scale analysis is more robust. If they move around substantially, protocol choice is itself a meaningful source of uncertainty.

## Notes after the numbers are in

- On the small three-sentence control example, all four protocols recovered the same diagonal sentence alignments as the strongest matches.
- The score magnitudes moved, but the ranking structure did not. `query_query` and `raw_raw` produced the highest maxima (`0.8143` and `0.8078`), while the two `query_corpus` directions were slightly lower (`0.7586` and `0.7636`).
- That means the asymmetry is real numerically, but for this clean toy case it did not alter the substantive answer.
- This is encouraging because it suggests protocol sensitivity may be smaller than content sensitivity for clearly parallel material.

This is worth knowing: a stable ranking on a toy example does not prove that protocol choice never matters. It only justifies moving on to messier real-corpus passages.


## Real-corpus takeaways from the saved run

- Across the four sampled cross-corpus passage pairs, `A1_SMDG_rig_pa_khu_byug` vs `B2_LL01_rtse_mo_byung_rgyal` was the strongest pair overall. Its max scores were `0.5916` (`A->B query_corpus`), `0.6242` (`B->A query_corpus`), `0.6488` (`query_query`), and `0.6556` (`raw_raw`).
- `A2_SMDG_sems_lung_rgyun_thag` vs `B2_LL01_rtse_mo_byung_rgyal` was the weakest pairing across most views, with maxima between `0.4420` and `0.5453`.
- `A1 vs B1` and `A2 vs B1` sat in the middle. Their maxima and upper-tail scores suggest real conceptual overlap, but less strongly than `A1 vs B2`.
- The broad ranking of the four text pairs stayed stable across protocols even though the absolute values shifted. That makes the sample analysis look methodologically usable rather than brittle.
- `query_query` consistently produced the highest mean and max scores. `raw_raw` was usually next. The retrieval-style `query_corpus` directions were lower, but they did not reverse the high-level ordering.

Interpretively, this looks more like the model is responding to a mix of rhetorical mode and doctrinal content than to a pure borrowing signal. The strongest pair also looked strongest to a human because both passages function like elevated doctrinal openings.


In [22]:
import numpy as np
import pandas as pd
import torch

from tibetan_pipeline import TibetanResearchSDK

print("device:", sdk.device)
print("torch_dtype:", sdk.torch_dtype)
print("load_in_8bit:", sdk.load_in_8bit)
print("batch_size:", sdk.batch_size)

PASSAGES = {
    "A1_SMDG_rig_pa_khu_byug": """སྟོན་པ་གཤེན་ལྷ་འོད་དཀར་ཐམས་ཅད་མཁྱེན་པའི་ཡེ་ཤེས་དང་ལྡན་པ་ལ་ཕྱག་འཚལ་ལོ། །སྟོན་པ་ཐུགས་རྗེ་ཆེར་ལྡན་བས། གསུང་ལས་སྤྲུལ་པའི་ཁུ་བྱུག་ནི། ཡུམ་ལ་ཆགས་ནས་དངོས་པོར་བྱུང་། དངོས་པོ་ཉིད་ལས་རྔོས་པོ་མྱེད། ཀུན་བཟང་ཐུགས་རྗེའི་སྐྱབས་འོག་ཏུ། དབྱིངས་ཀྱི་ཐིག་ལེ་གཅིག་ལ་ཐིམ། རིག་པ་ལྷའི་གསུང་དབྱངས་འདི། ཐབས་ཀྱིས་སྙན་ཁུངས་བརྒྱུད་པ་ཡིན། མི་གསང་གསང་བའི་གསང་མཆོག་འདི། བརྒྱ་ལམ་ན་ནི་ལན་འགའ་རུ། ཐོས་པ་དཀའ་ཡང་གོ་བར་སླ།""",
    "A2_SMDG_sems_lung_rgyun_thag": """སྟོན་པ་གཤེན་ལྷ་འོད་དཀར་གྱི་ཐུགས་ཡང་དག་པའི་བརྣག་པ། འབྲས་བུ་ངེས་པར་སྟོན་པའི་དོན། ཕྱིས་བྱོན་པའི་ལས་དག་པ་རྣམས་ལ་སྟོན་ཏེ། ཨེ་མ་ཧོ་གང་ལས་དག་པ་དག སེམས་མི་འགྱུར་བའི་གཡུང་དྲུང་འདི་བསྒོམ་པར་འདོད་ན། འདི་ལྟར་ཤེས་པར་བྱའོ་ཞེས། ཐུགས་རྗེས་བྱིན་གྱིས་བརླབས་ཀྱིས། གཤེན་ཚད་མེད་འོད་ལྡན་ལ་བསྟན་ནོ། །ལྟ་སྤྱོད་དམ་ཚིག་འཕྲིན་ལས་དག སྒོམ་པ་དག་ཏུ་ཤེས་པར་བྱ། མ་གཡོས་མ་བཅོས་གཉིས་སུ་མྱེད། རང་ལ་འཆར་གྱི་དྲོད་མྱོང་ནས། འབྲསབུ་དག་པའི་གདོན་མི་ཟའོ།""",
    "B1_L1_rig_pa_i_khu_byug": """དོག་དང་དབྱིབས་དང་མཚན་མ་མེད་པར་རིགས་རྒྱུད་དང་བྲལ་བ། གང་གི་གནས་སུ་མ་གྱུར་ཅིང་ཐམས་ཅད་ཀྱི་གཞིར་གྱུར་པ། མིང་དང་ཚིག་གིས་རྣམ་པར་དག་ཅིང་། རྣམ་པའི་སྤྱོད་པ་ཐམས་ཅད་དངས་ཤིང་བདེན་པ་གཉིས་མགོ་མཉམ་དུ་དྲིལ་པ། ཡེ་ཤེས་སྙིང་པོ་ཆེ་བ་ཐམས་ཅད་ཀྱི་བླ། ཆོས་ཐམས་ཅད་ནི་ཚུལ་བཞིན་གནས་ལ། ཆོས་ཐམས་ཅད་ཚུལ་བཞིན་དུ་མི་གནསཔའི་མཆོག །གང་བསམ་དུ་མེད་པའི་ངོ་བོ་ཉིད་""",
    "B2_LL01_rtse_mo_byung_rgyal": """རྩེ་མོ་བྱུང་རྒྱལ། བཅོམ་ལྡན་འདས་དཔལ་བདེ་བ་ཆེན་པོ་ཀུན་ཏུ་བཟང་པོ་ལ་ཕྱག་འཚལ་ལོ། །ཐམས་ཅད་མཁྱེན་པ་བདེ་བ་ཆེན་པོའི་ཐུགས། །སྣ་ཚོགས་ཀུན་ཏུ་བཟང་པོ་རྡོ་རྗེ་སེམས། །དབྱིངས་དང་ཡེ་ཤེས་འདུ་འབྲལ་མེད་པའི་སྐུ། །རྣམ་དག་བྱང་ཆུབ་སེམས་ལ་ཕྱག་འཚལ་ལོ། །སྔོན་ཚེ་འདས་པའི་བསྐལ་པའི་མཐའ་རིང་ནས། །འཁྲུལ་འཁོར་བླུན་རྨོངས་མདོངས་པའི་འགྲོ་བ་རྣམས། །ང་དང་བདག་བཅས་མཚན་མས་བཅིངས་པ་ཡིས། །ཁམས་གསུམ་རྒྱུད་ལས་འཁོར་བའི་འགྲོ་དོན་དུ། །སྙིགས་མའི་དོན་ལ་གདམས་པའི་གསུང་བརྗོད་འདི།""",
}

PAIR_CASES = [
    ("A1_SMDG_rig_pa_khu_byug", "B1_L1_rig_pa_i_khu_byug"),
    ("A1_SMDG_rig_pa_khu_byug", "B2_LL01_rtse_mo_byung_rgyal"),
    ("A2_SMDG_sems_lung_rgyun_thag", "B1_L1_rig_pa_i_khu_byug"),
    ("A2_SMDG_sems_lung_rgyun_thag", "B2_LL01_rtse_mo_byung_rgyal"),
]

def pairwise_protocol(protocol_name: str, sentences_a: list[str], sentences_b: list[str]):
    if protocol_name == "query_corpus":
        emb_a = sdk.embed_sentences(sentences_a, is_query=True)
        emb_b = sdk.embed_sentences(sentences_b, is_query=False)
    elif protocol_name == "query_query":
        emb_a = sdk.embed_sentences(sentences_a, is_query=True)
        emb_b = sdk.embed_sentences(sentences_b, is_query=True)
    elif protocol_name == "raw_raw":
        emb_a = sdk.embed_sentences(sentences_a, is_query=False)
        emb_b = sdk.embed_sentences(sentences_b, is_query=False)
    else:
        raise ValueError(protocol_name)
    return sdk.pairwise_from_embedding_views(
        emb_a,
        emb_b,
        top_k=min(12, len(sentences_a) * len(sentences_b)),
    )

def summarize_view(label: str, pair_name: str, view) -> dict[str, object]:
    matrix = view.similarity_matrix
    return {
        "pair": pair_name,
        "protocol": label,
        "shape": tuple(matrix.shape),
        "max_score": round(float(np.max(matrix)), 4) if matrix.size else None,
        "mean_score": round(float(np.mean(matrix)), 4) if matrix.size else None,
        "p95_score": round(float(np.percentile(matrix, 95)), 4) if matrix.size else None,
        "mean_best_row": round(float(np.mean(np.max(matrix, axis=1))), 4) if matrix.size else None,
        "mean_best_col": round(float(np.mean(np.max(matrix, axis=0))), 4) if matrix.size else None,
    }

def matrix_frame(view, row_label: str, col_label: str) -> pd.DataFrame:
    return pd.DataFrame(
        view.similarity_matrix,
        index=[f"{row_label}{i:02d}" for i in range(len(view.segments_a))],
        columns=[f"{col_label}{j:02d}" for j in range(len(view.segments_b))],
    )

def segment_passage(text: str) -> list[str]:
    seg = sdk.segment_text(text)
    return [s for s in seg.segments if s.strip()]

all_summaries = []
all_results = {}

for left_key, right_key in PAIR_CASES:
    left_text = PASSAGES[left_key]
    right_text = PASSAGES[right_key]
    pair_name = f"{left_key}__{right_key}"

    segments_a = segment_passage(left_text)
    segments_b = segment_passage(right_text)

    print(f"\n===== {pair_name} =====")
    print("A segments:", len(segments_a), "B segments:", len(segments_b))
    display(pd.DataFrame({"A_segments": pd.Series(segments_a), "B_segments": pd.Series(segments_b)}))

    view_ab = pairwise_protocol("query_corpus", segments_a, segments_b)
    view_ba = pairwise_protocol("query_corpus", segments_b, segments_a)
    view_qq = pairwise_protocol("query_query", segments_a, segments_b)
    view_rr = pairwise_protocol("raw_raw", segments_a, segments_b)

    pair_summary = pd.DataFrame(
        [
            summarize_view("A->B query_corpus", pair_name, view_ab),
            summarize_view("B->A query_corpus", pair_name, view_ba),
            summarize_view("A<->B query_query", pair_name, view_qq),
            summarize_view("A<->B raw_raw", pair_name, view_rr),
        ]
    )
    display(pair_summary)
    all_summaries.append(pair_summary)

    print("A->B query_corpus")
    display(matrix_frame(view_ab, "A", "B").round(4))
    print("B->A query_corpus")
    display(matrix_frame(view_ba, "B", "A").round(4))
    print("A<->B query_query")
    display(matrix_frame(view_qq, "A", "B").round(4))
    print("A<->B raw_raw")
    display(matrix_frame(view_rr, "A", "B").round(4))

    topk_tables = {
        "A->B query_corpus": view_ab.topk_dataframe(),
        "B->A query_corpus": view_ba.topk_dataframe(),
        "A<->B query_query": view_qq.topk_dataframe(),
        "A<->B raw_raw": view_rr.topk_dataframe(),
    }

    for label, table in topk_tables.items():
        print(f"## {pair_name} | {label}")
        display(table.head(10))

    all_results[pair_name] = {
        "segments_a": segments_a,
        "segments_b": segments_b,
        "view_ab": view_ab,
        "view_ba": view_ba,
        "view_qq": view_qq,
        "view_rr": view_rr,
    }

summary_df = pd.concat(all_summaries, ignore_index=True)
print("\n===== Combined Summary =====")
display(summary_df)

pivot_max = summary_df.pivot(index="pair", columns="protocol", values="max_score")
pivot_mean = summary_df.pivot(index="pair", columns="protocol", values="mean_score")

print("Max-score comparison")
display(pivot_max)

print("Mean-score comparison")
display(pivot_mean)


device: cuda
torch_dtype: float16
load_in_8bit: True
batch_size: 1

===== A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug =====
A segments: 12 B segments: 7


,A_segments,B_segments
0,སྟོན་པ་གཤེན་ལྷ་འོད་དཀར་ཐམས་ཅད་མཁྱེན་པའི་ཡེ་ཤེས...,དོག་དང་དབྱིབས་དང་མཚན་མ་མེད་པར་རིགས་རྒྱུད་དང་བྲ...
1,སྟོན་པ་ཐུགས་རྗེ་ཆེར་ལྡན་བས།,གང་གི་གནས་སུ་མ་གྱུར་ཅིང་ཐམས་ཅད་ཀྱི་གཞིར་གྱུར་པ།
2,གསུང་ལས་སྤྲུལ་པའི་ཁུ་བྱུག་ནི།,མིང་དང་ཚིག་གིས་རྣམ་པར་དག་ཅིང་། རྣམ་པའི་སྤྱོད་པ...
3,ཡུམ་ལ་ཆགས་ནས་དངོས་པོར་བྱུང་།,ཡེ་ཤེས་སྙིང་པོ་ཆེ་བ་ཐམས་ཅད་ཀྱི་བླ།
4,དངོས་པོ་ཉིད་ལས་རྔོས་པོ་མྱེད།,ཆོས་ཐམས་ཅད་ནི་ཚུལ་བཞིན་གནས་ལ།
5,ཀུན་བཟང་ཐུགས་རྗེའི་སྐྱབས་འོག་ཏུ།,ཆོས་ཐམས་ཅད་ཚུལ་བཞིན་དུ་མི་གནསཔའི་མཆོག །
6,དབྱིངས་ཀྱི་ཐིག་ལེ་གཅིག་ལ་ཐིམ།,གང་བསམ་དུ་མེད་པའི་ངོ་བོ་ཉིད་
7,རིག་པ་ལྷའི་གསུང་དབྱངས་འདི།,NaN
8,ཐབས་ཀྱིས་སྙན་ཁུངས་བརྒྱུད་པ་ཡིན།,NaN
9,མི་གསང་གསང་བའི་གསང་མཆོག་འདི།,NaN


[embed] backend=gemma-last-token device=cuda total_sentences=12 batch_size=1 batches=12
[embed] batch 1/12 sentences 1-1/12
[embed] batch 2/12 sentences 2-2/12
[embed] batch 3/12 sentences 3-3/12
[embed] batch 4/12 sentences 4-4/12
[embed] batch 5/12 sentences 5-5/12
[embed] batch 6/12 sentences 6-6/12
[embed] batch 7/12 sentences 7-7/12
[embed] batch 8/12 sentences 8-8/12
[embed] batch 9/12 sentences 9-9/12
[embed] batch 10/12 sentences 10-10/12
[embed] batch 11/12 sentences 11-11/12
[embed] batch 12/12 sentences 12-12/12
[embed] backend=gemma-last-token device=cuda total_sentences=7 batch_size=1 batches=7
[embed] batch 1/7 sentences 1-1/7
[embed] batch 2/7 sentences 2-2/7
[embed] batch 3/7 sentences 3-3/7
[embed] batch 4/7 sentences 4-4/7
[embed] batch 5/7 sentences 5-5/7
[embed] batch 6/7 sentences 6-6/7
[embed] batch 7/7 sentences 7-7/7
[embed] backend=gemma-last-token device=cuda total_sentences=7 batch_size=1 batches=7
[embed] batch 1/7 sentences 1-1/7
[embed] batch 2/7 sentences

,pair,protocol,shape,max_score,mean_score,p95_score,mean_best_row,mean_best_col
0,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A->B query_corpus,"(12, 7)",0.4867,0.3303,0.4507,0.4158,0.4178
1,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,B->A query_corpus,"(7, 12)",0.5039,0.3313,0.4447,0.4311,0.4171
2,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A<->B query_query,"(12, 7)",0.5814,0.4068,0.5367,0.5024,0.4955
3,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A<->B raw_raw,"(12, 7)",0.5230,0.3589,0.4887,0.4479,0.4575


A->B query_corpus


...

,pair,protocol,shape,max_score,mean_score,p95_score,mean_best_row,mean_best_col
0,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A->B query_corpus,"(12, 7)",0.4867,0.3303,0.4507,0.4158,0.4178
1,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,B->A query_corpus,"(7, 12)",0.5039,0.3313,0.4447,0.4311,0.4171
2,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A<->B query_query,"(12, 7)",0.5814,0.4068,0.5367,0.5024,0.4955
3,A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,A<->B raw_raw,"(12, 7)",0.5230,0.3589,0.4887,0.4479,0.4575
4,A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung...,A->B query_corpus,"(12, 9)",0.5916,0.3166,0.4671,0.4580,0.4589
5,A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung...,B->A query_corpus,"(9, 12)",0.6242,0.3195,0.4598,0.4556,0.4386
6,A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung...,A<->B query_query,"(12, 9)",0.6488,0.4017,0.5609,0.5455,0.5339
7,A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung...,A<->B raw_raw,"(12, 9)",0.6556,0.3454,0.5109,0.4807,0.4922
8,A2_SMDG_sems_lung_rgyun_thag__B1_L1_rig_pa_i_k...,A->B query_corpus,"(9, 7)",0.4882,0.3290,0.4407,0.4121,0.4306
9,A2_SMDG_sems_lung_rgyun_thag__B1_L1_rig_pa_i_k...,B->A query_corpus,"(7, 9)",0.4874,0.3444,0.4427,0.4316,0.4267


Max-score comparison


protocol,A->B query_corpus,A<->B query_query,A<->B raw_raw,B->A query_corpus
pair,,,,
A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,0.4867,0.5814,0.5230,0.5039
A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung_rgyal,0.5916,0.6488,0.6556,0.6242
A2_SMDG_sems_lung_rgyun_thag__B1_L1_rig_pa_i_khu_byug,0.4882,0.5594,0.5211,0.4874
A2_SMDG_sems_lung_rgyun_thag__B2_LL01_rtse_mo_byung_rgyal,0.4853,0.5453,0.5015,0.4420


Mean-score comparison


protocol,A->B query_corpus,A<->B query_query,A<->B raw_raw,B->A query_corpus
pair,,,,
A1_SMDG_rig_pa_khu_byug__B1_L1_rig_pa_i_khu_byug,0.3303,0.4068,0.3589,0.3313
A1_SMDG_rig_pa_khu_byug__B2_LL01_rtse_mo_byung_rgyal,0.3166,0.4017,0.3454,0.3195
A2_SMDG_sems_lung_rgyun_thag__B1_L1_rig_pa_i_khu_byug,0.3290,0.4040,0.3601,0.3444
A2_SMDG_sems_lung_rgyun_thag__B2_LL01_rtse_mo_byung_rgyal,0.3135,0.3927,0.3435,0.3270


## What the numbers justify doing next

- The notebook supports using the SDK as the inner engine for larger corpus analysis.
- It does **not** justify collapsing the corpus question to one similarity score or treating top-k matches as direct proof of copying.
- The next runner should save one sentence-embedding artifact per file, one `N x M` matrix per text pair, and a summary table with at least `max`, `p95`, `mean_best_row`, `mean_best_col`, and a top-k aggregate.
- At corpus scale, the main statistical question becomes whether the strongest cross-corpus pairs are exceptional relative to the background distribution, not whether any one raw score merely looks high.
- The most valuable human follow-up is to inspect whether the top-ranked sentence pairs are generic doctrinal-opening language, or whether they contain more specific and historically informative conceptual or phrase-level overlap.
